In [4]:

# ============================================================
# PUNJABI NEWS CLASSIFICATION
# SEED 42 — CPU INFERENCE — COMPLETE ONE-CELL CODE
# ============================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

from pathlib import Path
import json
import pickle
import re

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import tokenizer_from_json

# ============================================================
# 1. PATHS — YOUR ORIGINAL KAGGLE DATASET
# ============================================================

ROOT = Path(
    "/kaggle/input/datasets/drgurpreetsidhu/mymodel/"
    "HySCF_SixFusion_5Seed_Components"
)

MODEL_DIR = (
    ROOT / "experiments" /
    "F06_PARALLEL_FLAT_CNN__seed42"
)

PREPROCESSORS_DIR = ROOT / "preprocessors"

MODEL_PATH = MODEL_DIR / "model.keras"
TOKENIZER_PATH = PREPROCESSORS_DIR / "tokenizer.json"
TFIDF_PATH = (
    PREPROCESSORS_DIR /
    "tfidf_5000" /
    "vectorizer.pkl"
)
LABEL_PATH = PREPROCESSORS_DIR / "label_mapping.json"

# ============================================================
# 2. VERIFY FILES
# ============================================================

files = {
    "Model": MODEL_PATH,
    "Tokenizer": TOKENIZER_PATH,
    "TF-IDF": TFIDF_PATH,
    "Labels": LABEL_PATH
}

print("TensorFlow:", tf.__version__)
print("\nCHECKING FILES")

for name, path in files.items():
    print(name, ":", "FOUND" if path.is_file() else "MISSING")
    print(" ", path)

missing = [str(p) for p in files.values() if not p.is_file()]

if missing:
    raise FileNotFoundError(
        "Missing files:\n" + "\n".join(missing)
    )

# ============================================================
# 3. LOAD SAVED PREPROCESSORS
# ============================================================

print("\nLoading tokenizer...")
with open(TOKENIZER_PATH, encoding="utf-8") as f:
    tokenizer = tokenizer_from_json(f.read())

print("Loading fitted TF-IDF vectorizer...")
# Only load trusted pickle files from your training run.
with open(TFIDF_PATH, "rb") as f:
    vectorizer = pickle.load(f)

print("Loading labels...")
with open(LABEL_PATH, encoding="utf-8") as f:
    classes = json.load(f)["classes"]

# ============================================================
# 4. LOAD TRAINED MODEL
# ============================================================

print("Loading trained model...")

with tf.device("/CPU:0"):
    model = tf.keras.models.load_model(
        MODEL_PATH,
        compile=False
    )

assert model.count_params() == 3_417_100, (
    f"Unexpected parameter count: {model.count_params()}"
)

assert len(classes) == 10

input_names = [
    x.name.split(":")[0] for x in model.inputs
]

assert set(input_names) == {
    "han_input",
    "tfidf_input"
}, f"Unexpected inputs: {input_names}"

print("\nMODEL LOADED SUCCESSFULLY")
print("Parameters:", model.count_params())
print("Inputs:", input_names)
print("Categories:", classes)

# ============================================================
# 5. ORIGINAL TRAINING PREPROCESSING
# ============================================================

def preprocess_news(title, content):

    text = (
        str(title).strip() + " " +
        str(content).strip()
    ).strip()

    if not text:
        raise ValueError("Enter Punjabi news text.")

    # Contextual/CNN input: 15 segments x 30 tokens
    x_han = np.zeros(
        (1, 15, 30),
        dtype=np.int32
    )

    segments = re.split(r"[।!?]", text)[:15]
    sequences = tokenizer.texts_to_sequences(segments)

    for j, tokens in enumerate(sequences):
        tokens = tokens[:30]
        x_han[0, j, :len(tokens)] = tokens

    # Full-article TF-IDF: 5000 features
    x_tfidf = vectorizer.transform([text])
    x_tfidf = x_tfidf.toarray().astype(np.float32)

    if x_tfidf.shape != (1, 5000):
        raise ValueError(
            f"Expected TF-IDF shape (1,5000), got {x_tfidf.shape}"
        )

    return {
        "han_input": x_han,
        "tfidf_input": x_tfidf
    }

# ============================================================
# 6. PREDICT PUNJABI NEWS CATEGORY
# ============================================================

def predict_news(title, content):

    data = preprocess_news(title, content)

    inputs = [data[name] for name in input_names]

    with tf.device("/CPU:0"):
        probabilities = model.predict(
            inputs,
            verbose=0
        )[0]

    probabilities = np.asarray(
        probabilities,
        dtype=np.float32
    )

    order = np.argsort(probabilities)[::-1]
    predicted_index = int(order[0])

    print("\n" + "=" * 45)
    print("PUNJABI NEWS CLASSIFICATION")
    print("=" * 45)

    print("\nHeadline:", title)

    print("\nPredicted category:",
          classes[predicted_index])

    print("Model confidence:",
          f"{probabilities[predicted_index] * 100:.2f}%")

    print("\nTop 5 category probabilities:")

    for idx in order[:5]:
        idx = int(idx)
        print(
            f"{str(classes[idx]):18s}"
            f"{probabilities[idx] * 100:.2f}%"
        )

    return {
        "predicted_category": classes[predicted_index],
        "confidence": float(probabilities[predicted_index]),
        "probabilities": {
            str(classes[i]): float(probabilities[i])
            for i in range(10)
        }
    }

# ============================================================
# 7. TEST WITH PUNJABI ARTICLE
# ============================================================

TITLE = "ਪੰਜਾਬ ਵਿੱਚ ਕਿਸਾਨਾਂ ਲਈ ਨਵੀਂ ਖੇਤੀਬਾੜੀ ਯੋਜਨਾ ਦਾ ਐਲਾਨ"

CONTENT = """
ਪੰਜਾਬ ਸਰਕਾਰ ਨੇ ਕਿਸਾਨਾਂ ਦੀ ਮਦਦ ਲਈ
ਨਵੀਂ ਖੇਤੀਬਾੜੀ ਯੋਜਨਾ ਦਾ ਐਲਾਨ ਕੀਤਾ ਹੈ।
ਇਸ ਯੋਜਨਾ ਅਧੀਨ ਕਿਸਾਨਾਂ ਨੂੰ ਆਰਥਿਕ
ਸਹਾਇਤਾ ਅਤੇ ਖੇਤੀਬਾੜੀ ਨਾਲ ਸਬੰਧਤ
ਸਹੂਲਤਾਂ ਪ੍ਰਦਾਨ ਕੀਤੀਆਂ ਜਾਣਗੀਆਂ।
"""

result = predict_news(TITLE, CONTENT)


TensorFlow: 2.21.0

CHECKING FILES
Model : FOUND
  /kaggle/input/datasets/drgurpreetsidhu/mymodel/HySCF_SixFusion_5Seed_Components/experiments/F06_PARALLEL_FLAT_CNN__seed42/model.keras
Tokenizer : FOUND
  /kaggle/input/datasets/drgurpreetsidhu/mymodel/HySCF_SixFusion_5Seed_Components/preprocessors/tokenizer.json
TF-IDF : FOUND
  /kaggle/input/datasets/drgurpreetsidhu/mymodel/HySCF_SixFusion_5Seed_Components/preprocessors/tfidf_5000/vectorizer.pkl
Labels : FOUND
  /kaggle/input/datasets/drgurpreetsidhu/mymodel/HySCF_SixFusion_5Seed_Components/preprocessors/label_mapping.json

Loading tokenizer...
Loading fitted TF-IDF vectorizer...
Loading labels...
Loading trained model...


E0000 00:00:1793787590.286349      49 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)



MODEL LOADED SUCCESSFULLY
Parameters: 3417100
Inputs: ['han_input', 'tfidf_input']
Categories: ['agriculture', 'auto', 'business', 'education', 'entertainment', 'lifestyle', 'politics', 'religion', 'sports', 'technology']

PUNJABI NEWS CLASSIFICATION

Headline: ਪੰਜਾਬ ਵਿੱਚ ਕਿਸਾਨਾਂ ਲਈ ਨਵੀਂ ਖੇਤੀਬਾੜੀ ਯੋਜਨਾ ਦਾ ਐਲਾਨ

Predicted category: agriculture
Model confidence: 73.71%

Top 5 category probabilities:
agriculture       73.71%
religion          4.65%
technology        4.05%
business          3.51%
education         3.20%
